<img src=https://www.factset.com/hubfs/Assets/images/factset-logo.svg width="300" align="left">


# FactSet Supply Chain API Example - Danish Companies

Extract supply chain relationships (Suppliers, Customers, Competitors, Partners) for major Danish listed companies.

| Company | ISIN |
| --- | --- |
| Novo Nordisk B | DK0062498333 |
| DSV | DK0060079531 |
| Danske Bank | DK0010274414 |
| Vestas Wind Systems | DK0061539921 |
| Ørsted | DK0060094928 |
| Carlsberg B | DK0010181759 |
| A.P. Møller - Mærsk A | DK0010244425 |
| A.P. Møller - Mærsk B | DK0010244508 |
| Genmab | DK0010272202 |
| Coloplast | DK0060448595 |
| Tryg | DK0060636678 |
| Pandora | DK0060252690 |

## 1. Setup - Import packages and load credentials

In [ ]:
import requests
import json
import time
import pandas as pd
from requests.packages.urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)
from pandas import json_normalize

import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
USERNAME = os.getenv("USERNAME")
APIKEY = os.getenv("APIKEY")
SUPPLY_CHAIN_URL = 'https://api.factset.com/content/factset-supply-chain/v1/relationships'
DATA_PATH = "/Users/pg/Library/CloudStorage/Dropbox-CBS/Pipe Galera/fastest_data"

authorization = (USERNAME, APIKEY)
headers = {'Accept': 'application/json', 'Content-Type': 'application/json'}

## 2. Define Danish companies (ISINs)

In [ ]:
danish_companies = {
    "Novo Nordisk B": "DK0062498333",
    "DSV": "DK0060079531",
    "Danske Bank": "DK0010274414",
    "Vestas Wind Systems": "DK0061539921",
    "Ørsted": "DK0060094928",
    "Carlsberg B": "DK0010181759",
    "A.P. Møller - Mærsk A": "DK0010244425",
    "A.P. Møller - Mærsk B": "DK0010244508",
    "Genmab": "DK0010272202",
    "Coloplast": "DK0060448595",
    "Tryg": "DK0060636678",
    "Pandora": "DK0060252690",
}

In [ ]:
company_ids = list(danish_companies.values())
company_names = {v: k for k, v in danish_companies.items()}

print(f"Companies to query: {len(danish_companies)}")
for name, isin in danish_companies.items():
    print(f"  {name:30s} {isin}")

## Helper functions

In [ ]:
def reorder_columns(df, leading_cols=("requestCompany", "requestId")):
    """Move leading_cols to the front of the DataFrame, keeping the rest in original order."""
    front = [c for c in leading_cols if c in df.columns]
    rest = [c for c in df.columns if c not in front]
    return df[front + rest]


def fetch_relationships(ids, relationship_type, company_type="ALL", direction="ALL"):
    """Fetch supply chain relationships and return a cleaned DataFrame."""
    request_body = {
        "data": {
            "ids": ids if isinstance(ids, list) else [ids],
            "relationshipType": relationship_type,
            "companyType": company_type,
            "relationshipDirection": direction
        }
    }

    response = requests.post(
        url=SUPPLY_CHAIN_URL,
        data=json.dumps(request_body),
        auth=authorization,
        headers=headers,
        verify=False
    )
    print(f"{relationship_type:15s} - HTTP Status: {response.status_code}", end="")

    if response.status_code != 200:
        print(f" - Error: {response.text[:200]}")
        return pd.DataFrame()

    df = json_normalize(response.json()['data'])
    df['requestCompany'] = df['requestId'].map(company_names)
    df['relationshipType'] = relationship_type
    df = reorder_columns(df)
    print(f" - {len(df)} records")
    time.sleep(0.15)
    return df


def save_file(dataframe, type_name):
    """Save a DataFrame to CSV in the data directory."""
    file_name = f"danish_top_companies_{type_name}.csv"
    dataframe.to_csv(f"{DATA_PATH}/{file_name}", index=False)
    print(f"Saved: {file_name}")

## 3. Query all relationship types for all companies

The Supply Chain API accepts up to 500 IDs per request, so we can send all companies in a single call per relationship type. We query all four types: **Suppliers**, **Customers**, **Competitors**, and **Partners**, and include both public and private companies with all relationship directions.

In [ ]:
relationship_types = ["SUPPLIERS", "CUSTOMERS", "COMPETITORS", "PARTNERS"]
all_results = {}

for rel_type in relationship_types:
    all_results[rel_type] = fetch_relationships(company_ids, rel_type)

print(f"\nTotal records across all relationship types: {sum(len(df) for df in all_results.values())}")

## 4. Combine into a single DataFrame and explore

In [ ]:
# Combine all relationship types into one DataFrame
combined_df = pd.concat(all_results.values(), ignore_index=True)
combined_df = reorder_columns(combined_df, leading_cols=("requestCompany", "requestId", "relationshipType", "companyName"))

print(f"Combined DataFrame: {len(combined_df)} rows x {len(combined_df.columns)} columns\n")
print(f"Records per relationship type:")
print(combined_df['relationshipType'].value_counts().to_string())
display(combined_df.head(10))

In [ ]:
save_file(combined_df, "relationships")

## 5. Summary per company - relationship counts

In [ ]:
# Pivot: count of relationships per company and type
summary = combined_df.groupby(['requestCompany', 'relationshipType']).size().unstack(fill_value=0)
summary['TOTAL'] = summary.sum(axis=1)
summary = summary.sort_values('TOTAL', ascending=False)
summary = summary.reset_index().rename(columns={"requestCompany": "COMPANY"})
print("Relationship counts per company:\n")
summary

In [ ]:
save_file(summary, "relationships_summary")

## 6. Detailed view per relationship type

In [ ]:
# Display each relationship type sorted by overlap percentage
for rel_type in relationship_types:
    df = all_results.get(rel_type)
    if df is not None and not df.empty:
        display_df = df[['requestCompany', 'companyName', 'overlappingProductCount',
                         'overlapPercentage', 'relationshipDirection']].copy()
        display_df = display_df.sort_values(
            ['requestCompany', 'overlapPercentage'], ascending=[True, False]
        )
        print(f"\n{'='*80}")
        print(f"  {rel_type} ({len(display_df)} records)")
        print(f"{'='*80}")
        display(display_df.reset_index(drop=True))
    else:
        print(f"\n{rel_type}: No data returned")